In [1]:
from pathlib import Path

PREDICT_SAMPLES_DIR = Path('./pred_samples_v1')

OUTPUT_DIR = Path('output/v1')
OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
import os
import gc
import glob
import rasterio
import numpy as np
import tensorflow as tf

from tqdm import tqdm
from sys import getsizeof
from datetime import datetime
from patchify import patchify
from rasterio.transform import Affine

I0000 00:00:1763066301.329360  148292 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1763066301.329754  148292 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1763066301.357997  148292 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1763066302.973089  148292 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENAB

In [3]:
import os

# 1. Desabilitar o XLA (Geralmente a causa raiz em hardware muito novo com TF)
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
# Opcional: Se o acima não bastar, force o desligamento do JIT
os.environ['TF_JIT_PROFILING'] = 'false'

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Memory Growth habilitado para: {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"Erro ao configurar memória: {e}")

E0000 00:00:1763066303.527039  148292 cuda_executor.cc:1695] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1763066303.527248  148338 cuda_executor.cc:1713] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1763066303.542531  148292 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0000 00:00:1763066303.542769  148292 gpu_device.cc:2362] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping regis

In [4]:
model = tf.keras.models.load_model('my_model.keras')

model.summary()

Model: "extractor_until_conv2d_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 6_band_input (InputLayer)       │ (None, 256, 256, 6)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_first_conv (Conv2D)         │ (None, 128, 128, 16)   │           880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_tail (Functional)     │ (None, 8, 8, 960)      │     2,995,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d (UpSampling2D)    │ (None, 16, 16, 960)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 16, 16, 512)    │     4,424,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_1 (UpSampling2D)  │ (None, 32, 32, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 256)    │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_2 (UpSampling2D)  │ (None, 64, 64, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │       295,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,895,936 (33.94 MB)

 Trainable params: 8,871,536 (33.84 MB)

 Non-trainable params: 24,400 (95.31 KB)

In [5]:
import rasterio
import numpy as np
import tensorflow as tf
from rasterio.transform import Affine
from patchify import patchify
import os
from datetime import datetime
from sys import getsizeof
from tqdm import tqdm

def generate_embedding_mosaic(mosaic_path, year, region_id, version, output_dir, model, EPOCH=100, patch_size=256, step=128):
    """
    Generates a multi-band embedding/feature mosaic from a large image using a
    smart, flexible, and memory-efficient stitching process with overlap-averaging.
    """
    # --- Sections 1-4 are correct and remain the same ---
    output_dir_with_epoch = os.path.join(output_dir, str(year), str(EPOCH))
    os.makedirs(output_dir_with_epoch, exist_ok=True)
    base_filename = os.path.basename(mosaic_path).replace('.tif', '')
    final_raster_uri = os.path.join(output_dir_with_epoch, f'outimage_v{version}_e{EPOCH}_grid_{region_id}_{year}_embedding_normalized.tif')
    if os.path.exists(final_raster_uri):
        print(f"Final file already exists, skipping: {final_raster_uri}")
        return
    print(f"--- Starting embedding generation for model: '{model.name}' ---")
    with rasterio.open(mosaic_path, 'r') as ds:
        out_meta = ds.meta.copy()
        original_transform = ds.transform
        img_arr = ds.read().astype(np.float32)
    img_arr = np.nan_to_num(np.clip(img_arr, 0, None), nan=0.0)
    if img_arr.shape[0] > 6:
        img_arr = img_arr[1:, :, :]
    img_arr_hwc = np.transpose(img_arr, [1, 2, 0])
    h_original, w_original, _ = img_arr_hwc.shape
    print(f"Original image shape (H, W, C): {(h_original, w_original, 6)}")
    #NP.pad is missing
    patches = patchify(img_arr_hwc, (patch_size, patch_size, 6), step=step) #


    patches_reshaped = patches.reshape(-1, patch_size, patch_size, 6)
    print(f"Image patched into {patches_reshaped.shape[0]} patches.")
    print("Running memory-efficient prediction...")
    patch_dataset = tf.data.Dataset.from_tensor_slices(patches_reshaped).batch(64)
    prediction_list = [model.predict_on_batch(batch) for batch in tqdm(patch_dataset, desc="Predicting Batches")]
    predictions = np.concatenate(prediction_list, axis=0)
    print("Prediction complete. Predictions shape:", predictions.shape)

    # --- Section 5 is correct and remains the same ---
    print("Stitching results with smart scaling and overlap-averaging...")
    grid_rows, grid_cols = patches.shape[0], patches.shape[1]
    _, pred_h, pred_w, pred_c = predictions.shape
    scale_h, scale_w = patch_size / pred_h, patch_size / pred_w
    pred_step_h, pred_step_w = int(step / scale_h), int(step / scale_w)
    assert step % scale_h == 0 and step % scale_w == 0, "Step size must be divisible by scaling factor."
    canvas_h, canvas_w = int(h_original / scale_h), int(w_original / scale_w)
    canvas_shape = (canvas_h, canvas_w, pred_c)
    prediction_canvas = np.zeros(canvas_shape, dtype=np.float32)
    overlap_counter = np.zeros(canvas_shape, dtype=np.int32)
    for k, predicted_patch in enumerate(tqdm(predictions, desc="Stitching Patches")):
        # predicted_patch[0, :, :] = 1;
        # predicted_patch[-1, :, :] = 1
        # predicted_patch[:, 0, :] = 1;
        # predicted_patch[:, -1, :] = 1
        i = k // grid_cols
        j = k % grid_cols
        x_start, y_start = i * pred_step_h, j * pred_step_w
        prediction_canvas[x_start:x_start + pred_h, y_start:y_start + pred_w] += predicted_patch
        overlap_counter[x_start:x_start + pred_h, y_start:y_start + pred_w] += 1
    print("Averaging overlapped regions...")
    overlap_counter[overlap_counter == 0] = 1
    smooth_mosaic_scaled = prediction_canvas / overlap_counter

    # --- 6. CORRECTED: Resize, Normalize, and Save ---

    print("Normalizing final mosaic...")
    min_vals = np.min(smooth_mosaic_scaled, axis=(0, 1))
    max_vals = np.max(smooth_mosaic_scaled, axis=(0, 1))
    range_vals = max_vals - min_vals
    range_vals[range_vals == 0] = 1.0
    normalized_image = (smooth_mosaic_scaled - min_vals) / range_vals
    final_image_uint8 = (normalized_image * 255).astype(np.uint8)

    reconstructed_image_chw = np.transpose(final_image_uint8, [2, 0, 1])
    print("Stitching complete. Final array shape (C, H, W):", reconstructed_image_chw.shape)


    original_gdal_transform = out_meta['transform']
    new_transform = original_gdal_transform * Affine.scale(scale_w, scale_h)

    # --- 7. Save the Final GeoTIFF (This part is correct) ---
    out_meta.update({
    "driver": "GTiff",
    "height": smooth_mosaic_scaled.shape[0],  # Use the SMALL height
    "width": smooth_mosaic_scaled.shape[1],   # Use the SMALL width
    "count": reconstructed_image_chw.shape[0],   # The number of feature channels # BUG encontrado
    "dtype": "uint8",      # Use the float32 dtype of the mosaic
    "transform": new_transform,               # Use our NEW scaled transform
    "compress": 'lzw'
})


    print(f"Saving final output to: {final_raster_uri}")
    with rasterio.open(final_raster_uri, 'w', **out_meta, tiled=True, blockxsize=256, blockysize=256, predictor=2) as dest:
        dest.write(reconstructed_image_chw)

    print("Processing complete!")

In [6]:
def check_file_exists(paths):
    """Verifica se algum dos caminhos especificados já existe."""
    for path in paths:
        if os.path.exists(path):
            return True
    return False
def create_directory(new_folder):
  if not os.path.exists(new_folder):
      print(f'lets make the directory: {new_folder}')
      os.makedirs(new_folder)
  else: return

def dynamic_slice_or_pad(target_shape, predicted_img):
    print(f'ORIGINAL SHAPE: {target_shape}')
    print(f'predicted_img SHAPE: {predicted_img.shape}')
    target_rows, target_cols = target_shape
    pad_rows = target_rows - predicted_img.shape[0]
    pad_cols = target_cols - predicted_img.shape[1]

    # ORIGINAL ROWS IS SMALLER
    if pad_rows < 0:
        predicted_img = predicted_img[:target_rows, :]
    elif pad_rows > 0:
        predicted_img = np.pad(predicted_img, ((0, pad_rows), (0, 0)), mode='constant')

    # ORIGINAL COLUMNS IS SMALLER
    if pad_cols < 0:
        predicted_img = predicted_img[:, :target_cols]
    elif pad_cols > 0:
        predicted_img = np.pad(predicted_img, ((0, 0), (0, pad_cols)), mode='constant')


    print(f'predicted_img SHAPE FINALL  : {predicted_img.shape}')
    return predicted_img

In [ ]:
pattern = f"{PREDICT_SAMPLES_DIR}/*"
matching_files = glob.glob(pattern)

generate_embedding_mosaic(matching_files[2], 2020, 2, 1, OUTPUT_DIR, model, 100, 256, 32)
generate_embedding_mosaic(matching_files[5], 2020, 5, 1, OUTPUT_DIR, model, 100, 256, 32)

--- Starting embedding generation for model: 'extractor_until_conv2d_2' ---
Original image shape (H, W, C): (1856, 1856, 6)
Image patched into 10404 patches.
Running memory-efficient prediction...
